# Attention Mechanism

Calculate a small attention read by hand, compare it with PyTorch, and apply a causal mask. The examples use tiny tensors so every score and weight is inspectable.

In [2]:
import torch
from torch.nn import functional as F

torch.manual_seed(26)
torch.set_num_threads(1)
dtype = torch.float64
print('CPU | attention arithmetic')

CPU | attention arithmetic


## Work the weighted read

For query 1 and keys 0 and 1, the scores are [0, 1]. Softmax turns them into weights. Apply those weights to values 2 and 6.

In [3]:
scores = torch.tensor([[0., 1.]], dtype=dtype)
values = torch.tensor([[[2.], [6.]]], dtype=dtype)
weights = torch.softmax(scores, dim=-1)
manual_output = weights @ values
expected_weights = torch.tensor([[0.2689414213699951, 0.7310585786300049]], dtype=dtype)
torch.testing.assert_close(weights, expected_weights)
torch.testing.assert_close(manual_output.squeeze(), torch.tensor(4.9242343145200195, dtype=dtype))
print('weights:', weights.tolist())
print('weighted value:', manual_output.item())

weights: [[0.26894142136999516, 0.7310585786300049]]
weighted value: 4.92423431452002


## Calculate scaled dot-product attention

With vector queries and keys, dot products measure matches. Divide by the square root of the key width before softmax, then use the weights to combine values. PyTorch provides this operation directly.

In [ ]:
query = torch.tensor([[[[1., 0.]]]], dtype=dtype)
keys = torch.tensor([[[[1., 0.], [0., 1.]]]], dtype=dtype)
values = torch.tensor([[[[2., 0.], [0., 6.]]]], dtype=dtype)
scores = query @ keys.transpose(-2, -1) / (query.shape[-1] ** 0.5)
weights = torch.softmax(scores, dim=-1)
manual = weights @ values
built_in = F.scaled_dot_product_attention(query, keys, values)
torch.testing.assert_close(manual, built_in)
assert manual.shape == (1, 1, 1, 2)
print('scores:', scores.flatten().tolist())
print('weights:', weights.flatten().tolist())
print('output:', manual.flatten().tolist())

## See what a causal mask hides

A causal mask lets each position use itself and earlier positions but not the future. Compare the first output with and without the mask.

In [ ]:
sequence = torch.tensor([[[[1., 0.], [0., 1.]]]], dtype=dtype)
unmasked = F.scaled_dot_product_attention(sequence, sequence, sequence)
causal = F.scaled_dot_product_attention(sequence, sequence, sequence, is_causal=True)
torch.testing.assert_close(causal[..., 0, :], sequence[..., 0, :])
assert not torch.allclose(unmasked[..., 0, :], causal[..., 0, :])
print('unmasked first output:', unmasked[..., 0, :].flatten().tolist())
print('causal first output:', causal[..., 0, :].flatten().tolist())